# 01 — Data collection and provenance

Source inventory and provenance only. Inspect the immutable-source registry and validate representative current archives. This notebook makes no downloads and writes no data; reusable validation lives in `src/collection_validation.py`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_support import resolve_project_root

PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)

import pandas as pd
from IPython.display import display
from src.collection_validation import validate_icnf_archive, validate_zip_archive
from src.config import OPERATIONAL_FORECAST
from src.source_registry import CAOP_2025, ICNF_ANNUAL_ARCHIVES, REGISTERED_SOURCES

print('Project root resolved from the cloned repository.')
print(f'Registered immutable ZIP sources: {len(REGISTERED_SOURCES)}')
print('No automated downloads are executed.')

Project root resolved from the cloned repository.
Registered immutable ZIP sources: 19
No automated downloads are executed.


## Registered-source ledger

This table is generated from the immutable provenance registry. It makes the local file contract, source URL, acquisition method, and expected checksum inspectable without copying registry values into notebook prose.

In [2]:
icnf_ledger = pd.DataFrame([
    {
        'burned-area year': year,
        'raw path': record.raw_path,
        'filename': record.filename,
        'acquisition': record.acquisition_method,
        'registered CRS': record.validation_facts.crs if record.validation_facts else None,
        'expected features': record.validation_facts.feature_count if record.validation_facts else None,
    }
    for year, record in sorted(ICNF_ANNUAL_ARCHIVES.items())
])
display(icnf_ledger)
print('Official catalogue:', ICNF_ANNUAL_ARCHIVES[2025].official_source_url)
print('The 2000–2008 source is registered as a combined archive with separately validated annual layers.')

,burned-area year,raw path,filename,acquisition,registered CRS,expected features
0,2009,data/raw/wildfire/icnf_burned_areas/ardida_200...,ardida_2009.zip,pre-existing local archive; original acquisiti...,EPSG:3763,1441
1,2010,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2010.zip,pre-existing local archive; original acquisiti...,EPSG:3763,2513
2,2011,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2011.zip,pre-existing local archive; original acquisiti...,EPSG:3763,3686
3,2012,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2012.zip,manual browser download,EPSG:3763,2971
4,2013,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2013.zip,manual browser download,EPSG:3763,3150
5,2014,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2014.zip,manual browser download,EPSG:3763,1100
6,2015,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2015.zip,manual browser download,EPSG:3763,1651
7,2016,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2016.zip,manual browser download,EPSG:3763,2838
8,2017,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2017.zip,manual browser download,EPSG:3763,2765
9,2018,data/raw/wildfire/icnf_burned_areas/ardida_201...,ardida_2018.zip,manual browser download,EPSG:3763,537


Official catalogue: https://si.icnf.pt/shp/ardida_2025
The 2000–2008 source is registered as a combined archive with separately validated annual layers.


## Read-only archive checks

Validate the CAOP archive and the ICNF years needed at the current labelled/scoring boundary. Invalid ICNF geometries remain immutable source facts and are repaired only in derived processing.

In [3]:
print('CAOP:', validate_zip_archive(CAOP_2025, PROJECT_ROOT)['zip_integrity'])
for year in (2023, 2024, 2025):
    result = validate_icnf_archive(ICNF_ANNUAL_ARCHIVES[year], PROJECT_ROOT, expected_year=year)
    print(year, result['feature_count'], result['non_empty_geometry_count'], result['invalid_geometry_count'])

CAOP: passed


2023 1736 1736 11


2024 1558 1558 0


2025 2084 2084 2


## Current annual temporal roles

For forecast year 2026, predictor inputs are from T=2025. The model is labelled only through T=2024 / observed outcome 2025. Same-year ICNF burned area is never a predictor.

In [4]:
forecast_year = OPERATIONAL_FORECAST.current_forecast_year
predictor_year = OPERATIONAL_FORECAST.predictor_year(forecast_year)
history_years = OPERATIONAL_FORECAST.history_years(forecast_year)
assert max(history_years) < predictor_year
print({'forecast_year': forecast_year, 'predictor_year': predictor_year, 'history_years': history_years, 'latest_observed_outcome': OPERATIONAL_FORECAST.latest_observed_outcome_year(forecast_year)})

{'forecast_year': 2026, 'predictor_year': 2025, 'history_years': (2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024), 'latest_observed_outcome': 2025}
